In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp

lat = 53.336647
lon = 10.536493
api_key = dbutils.secrets.get(scope="prod-ingestion", key="opean-weather-default-key")
language = "de"
units = "metric"

url = f"https://api.openweathermap.org/data/2.5/weather"
params = {
    "lat" : lat,
    "lon" : lon,
    "appid" : api_key,
    "lang" : language,
    "units" : units
}

response = requests.get(url, params=params)
data = response.json()


In [0]:
import json
from pyspark.sql.functions import current_timestamp

json_str = json.dumps(data)

df = spark.createDataFrame(
    [(json_str,)],
    ["raw_json"]
).withColumn("ingestion_time", current_timestamp())

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
df.write.format("delta").mode("append").saveAsTable("workspace.default.bronze_weather")